# Notebook 10 — Production: The LeRobot Format, Policy Export, and the Path to Hardware

### Course roadmap

**Part I — the data** · 1. What an action really is · 2. Anatomy of a demonstration
**Part II — simulation** · 3. Physics from scratch · 4. The robot model · 5. System identification
**Part III — learning** · 6. Behavior cloning · 7. Three fixes · 8. RL in simulation
**Part IV — reality** · 9. Domain randomization · **10. Production & deployment**

---

### The real-world problem

> **Everything so far produced numbers in a notebook. Production needs artifacts other
> people and other machines can consume.**

This notebook closes the loop. Three concrete deliverables:

1. **Write a dataset in LeRobot v3 format** — the interchange format the whole ecosystem
   reads. Notebook 1 *consumed* it; here we *produce* it, which is what you do after
   collecting your own demonstrations.
2. **Export a trained policy** with everything needed to run it: weights, the
   normalization statistics, the identified dynamics it was trained against, and the
   observation contract.
3. **The hardware deployment checklist** — what you can verify before touching a robot,
   and what genuinely requires one.

### A dependency decision, stated up front

`pip install lerobot` into this environment **downgrades torch from 2.14 to 2.10**, which
would break notebooks 3-9. I checked:

```
- torch==2.14.1
+ torch==2.10.0
+ torchcodec==0.10.0
+ torchvision==0.25.0
```

So this notebook deliberately **does not install `lerobot`**. That is not a workaround —
it is the correct call, and it reflects how you would actually work: LeRobot gets its own
environment, and the two talk through *files*, which is exactly what a standard format is
for. Everything below runs with `pandas` + `pyarrow` + `torch`, and produces artifacts
LeRobot can read.

Section 5 gives the exact commands for the separate LeRobot environment and the hardware
path.

In [1]:
import json, shutil, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn as nn

STATS = json.load(open("../artifacts/so101_stats.json"))        # nb 2
IDENT = json.load(open("../artifacts/so101_identified.json"))   # nb 5
JOINTS, FPS = STATS["joints"], STATS["fps"]

EXPORT = Path("../export"); EXPORT.mkdir(exist_ok=True)
print(f"joints {JOINTS}")
print(f"fps {FPS} | identified kp {IDENT['identified_parameters']['kp']} | "
      f"measured lag {IDENT['validation']['real_lag_frames']} frames")
print(f"export dir -> {EXPORT.resolve()}")

joints ['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper']
fps 30 | identified kp 18.0 | measured lag 4 frames
export dir -> /Users/manujmehrotra/Desktop/repository/reinforcement-learning/physical-ai/export


## 1. Writing a LeRobot v3 dataset

Notebook 1 read this format. The structure, from `meta/info.json` onward:

```
data/chunk-000/file-000.parquet     low-dimensional signals: state, action, timestamps
videos/<camera>/chunk-000/*.mp4     camera streams
meta/info.json                      the schema: every feature, its dtype and shape
meta/episodes/...parquet            per-episode metadata and statistics
meta/tasks.parquet                  task descriptions
meta/stats.json                     dataset-level normalization statistics
```

The design is deliberate and worth restating: **Parquet for high-frequency
low-dimensional signals, MP4 for video.** Storing 480×640×3 frames in a dataframe would
be absurd, and storing joint angles in a video would be lossy.

We will write a small, schema-valid dataset from simulated rollouts. The point is not the
data — it is that **you can produce the format**, which is what lets your robot's
demonstrations join the ecosystem instead of living in a bespoke pickle.

In [2]:
def make_rollout(n_frames, seed=0):
    """Stand-in for teleoperation: a smooth joint trajectory plus the position targets
    that would have produced it. Uses the lag we measured in notebook 1, so the
    action/state relationship in this synthetic dataset matches the real arm's."""
    rng = np.random.default_rng(seed)
    t = np.arange(n_frames) / FPS
    amp = rng.uniform(10, 40, len(JOINTS))
    freq = rng.uniform(0.15, 0.4, len(JOINTS))
    phase = rng.uniform(0, 2 * np.pi, len(JOINTS))
    centre = np.array(STATS["observation.state"]["mean"], dtype=np.float32)
    state = centre + amp * np.sin(2 * np.pi * freq * t[:, None] + phase)
    lag = IDENT["validation"]["real_lag_frames"]          # 4 frames, from nb 1
    action = np.roll(state, -lag, axis=0)                 # the target the state chases
    action[-lag:] = state[-1]
    return state.astype(np.float32), action.astype(np.float32)

OUT = EXPORT / "so101_demo_dataset"
if OUT.exists(): shutil.rmtree(OUT)
(OUT / "data" / "chunk-000").mkdir(parents=True)
(OUT / "meta" / "episodes" / "chunk-000").mkdir(parents=True)

N_EPISODES, EP_LEN = 6, 90
rows, ep_meta, idx = [], [], 0
for e in range(N_EPISODES):
    st, ac = make_rollout(EP_LEN, seed=e)
    for f in range(EP_LEN):
        rows.append({
            "action": ac[f].tolist(),
            "observation.state": st[f].tolist(),
            "timestamp": float(f / FPS),
            "frame_index": f,
            "episode_index": e,
            "index": idx,
            "task_index": 0,
        })
        idx += 1
    ep_meta.append({"episode_index": e, "tasks": ["reach and return"],
                    "length": EP_LEN, "dataset_from_index": e * EP_LEN,
                    "dataset_to_index": (e + 1) * EP_LEN})

df = pd.DataFrame(rows)
pq.write_table(pa.Table.from_pandas(df, preserve_index=False),
               OUT / "data" / "chunk-000" / "file-000.parquet")
pq.write_table(pa.Table.from_pandas(pd.DataFrame(ep_meta), preserve_index=False),
               OUT / "meta" / "episodes" / "chunk-000" / "file-000.parquet")
pq.write_table(pa.Table.from_pandas(
    pd.DataFrame([{"task_index": 0, "task": "reach and return"}]), preserve_index=False),
    OUT / "meta" / "tasks.parquet")

info = {
    "codebase_version": "v3.0",
    "robot_type": "so100_follower",
    "fps": FPS,
    "total_episodes": N_EPISODES,
    "total_frames": len(df),
    "total_tasks": 1,
    "chunks_size": 1000,
    "data_path": "data/chunk-{chunk_index:03d}/file-{file_index:03d}.parquet",
    "video_path": "videos/{video_key}/chunk-{chunk_index:03d}/file-{file_index:03d}.mp4",
    "features": {
        "action": {"dtype": "float32", "shape": [len(JOINTS)],
                   "names": [f"{j}.pos" for j in JOINTS]},
        "observation.state": {"dtype": "float32", "shape": [len(JOINTS)],
                              "names": [f"{j}.pos" for j in JOINTS]},
        "timestamp": {"dtype": "float32", "shape": [1], "names": None},
        "frame_index": {"dtype": "int64", "shape": [1], "names": None},
        "episode_index": {"dtype": "int64", "shape": [1], "names": None},
        "index": {"dtype": "int64", "shape": [1], "names": None},
        "task_index": {"dtype": "int64", "shape": [1], "names": None},
    },
}
json.dump(info, open(OUT / "meta" / "info.json", "w"), indent=2)

A = np.stack(df["action"].values); S = np.stack(df["observation.state"].values)
json.dump({
    "action": {"mean": A.mean(0).tolist(), "std": A.std(0).tolist(),
               "min": A.min(0).tolist(), "max": A.max(0).tolist()},
    "observation.state": {"mean": S.mean(0).tolist(), "std": S.std(0).tolist(),
                          "min": S.min(0).tolist(), "max": S.max(0).tolist()},
}, open(OUT / "meta" / "stats.json", "w"), indent=2)

print(f"wrote {N_EPISODES} episodes / {len(df)} frames to {OUT.relative_to(EXPORT.parent)}")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size/1024:.1f} KB)")

wrote 6 episodes / 540 frames to export/so101_demo_dataset
  data/chunk-000/file-000.parquet  (56.6 KB)
  meta/episodes/chunk-000/file-000.parquet  (3.7 KB)
  meta/info.json  (1.4 KB)
  meta/stats.json  (1.4 KB)
  meta/tasks.parquet  (1.7 KB)


## 2. Validate it the way notebook 1 read it

Writing a format is easy; writing a *valid* one is the part that bites. So we validate by
**loading it back with exactly the code path from notebook 1** — read `meta/info.json`,
then the Parquet into pandas, then check the declared schema against the actual columns.

If this round-trip passes, the dataset is consumable by anything that reads the format.

In [3]:
def validate_lerobot_dataset(root):
    root = Path(root)
    problems, checks = [], []
    def chk(ok, msg):
        checks.append((ok, msg))
        if not ok: problems.append(msg)

    chk((root / "meta" / "info.json").exists(), "meta/info.json present")
    info = json.load(open(root / "meta" / "info.json"))
    for k in ["codebase_version", "fps", "total_episodes", "total_frames", "features"]:
        chk(k in info, f"info.json has '{k}'")

    pfile = root / "data" / "chunk-000" / "file-000.parquet"
    chk(pfile.exists(), "data parquet present")
    d = pd.read_parquet(pfile)
    chk(len(d) == info["total_frames"], f"row count {len(d)} == total_frames {info['total_frames']}")
    chk(d["episode_index"].nunique() == info["total_episodes"],
        f"episode count {d['episode_index'].nunique()} == total_episodes {info['total_episodes']}")

    # every declared non-video feature must be a real column, with the right width
    for name, feat in info["features"].items():
        if feat["dtype"] == "video":
            continue
        chk(name in d.columns, f"column '{name}' exists")
        if name in d.columns and feat["shape"][0] > 1:
            w = len(np.asarray(d[name].iloc[0]))
            chk(w == feat["shape"][0], f"'{name}' width {w} == declared {feat['shape'][0]}")

    # bookkeeping invariants
    chk(d["index"].is_monotonic_increasing, "global index monotonic")
    per_ep_ok = all(g["frame_index"].tolist() == list(range(len(g)))
                    for _, g in d.groupby("episode_index"))
    chk(per_ep_ok, "frame_index restarts at 0 within each episode")
    dt = np.diff(d[d.episode_index == 0]["timestamp"].values)
    chk(np.allclose(dt, 1 / info["fps"], atol=1e-6), f"timestamps spaced at 1/{info['fps']}s")
    chk((root / "meta" / "stats.json").exists(), "meta/stats.json present")
    chk((root / "meta" / "tasks.parquet").exists(), "meta/tasks.parquet present")

    for ok, msg in checks:
        print(f"  [{'OK ' if ok else 'FAIL'}] {msg}")
    return problems

print("validating the dataset we just wrote:\n")
probs = validate_lerobot_dataset(OUT)
print(f"\n{len(probs)} problems" if probs else "\nall checks passed")

# And the real test: does the nb-1 reading code work on it unchanged?
print("\nre-reading with notebook 1's code path:")
inf = json.load(open(OUT / "meta" / "info.json"))
dd = pd.read_parquet(OUT / "data" / "chunk-000" / "file-000.parquet")
Aa = np.stack(dd["action"].values).astype(np.float64)
Ss = np.stack(dd["observation.state"].values).astype(np.float64)
ep0 = dd[dd.episode_index == 0]
A0 = np.stack(ep0["action"].values).astype(np.float64)
S0 = np.stack(ep0["observation.state"].values).astype(np.float64)
errs = [np.abs(A0[:len(A0)-k] - S0[k:]).mean() for k in range(9)]
print(f"  robot {inf['robot_type']} | {inf['total_episodes']} eps | {inf['total_frames']} frames")
print(f"  recovered best lag: {int(np.argmin(errs))} frames "
      f"(we injected {IDENT['validation']['real_lag_frames']})")
print("  -> notebook 1's lag analysis runs on our dataset and recovers what we put in")

validating the dataset we just wrote:

  [OK ] meta/info.json present
  [OK ] info.json has 'codebase_version'
  [OK ] info.json has 'fps'
  [OK ] info.json has 'total_episodes'
  [OK ] info.json has 'total_frames'
  [OK ] info.json has 'features'
  [OK ] data parquet present
  [OK ] row count 540 == total_frames 540
  [OK ] episode count 6 == total_episodes 6
  [OK ] column 'action' exists
  [OK ] 'action' width 6 == declared 6
  [OK ] column 'observation.state' exists
  [OK ] 'observation.state' width 6 == declared 6
  [OK ] column 'timestamp' exists
  [OK ] column 'frame_index' exists
  [OK ] column 'episode_index' exists
  [OK ] column 'index' exists
  [OK ] column 'task_index' exists
  [OK ] global index monotonic
  [OK ] frame_index restarts at 0 within each episode
  [OK ] timestamps spaced at 1/30s
  [OK ] meta/stats.json present
  [OK ] meta/tasks.parquet present

all checks passed

re-reading with notebook 1's code path:
  robot so100_follower | 6 eps | 540 frames
  recovered

That last line is the real validation. Notebook 1's central experiment — sweep the lag,
find where `action[t]` best predicts `state[t+k]` — runs **unmodified** on a dataset we
wrote, and recovers the lag we injected. The format round-trips semantically, not just
structurally.

## 3. Exporting a policy so someone else can run it

A `state_dict` on its own is not a deployable artifact. To run a policy you need to know:

| | why it is required |
|---|---|
| **weights** | obviously |
| **observation contract** | what goes in, in what order, in what units |
| **normalization statistics** | notebook 2's; the wrong ones silently produce garbage |
| **action semantics** | notebook 1: these are *position targets*, not torques |
| **control rate** | notebook 1: 30 Hz. Running at 10 Hz changes the MDP |
| **dynamics it was trained against** | notebook 5's identified parameters |
| **known limitations** | what it was never trained to handle |

Leave any of these out and the person deploying it has to guess. Most of this course's
debugging time went into things that would have been in such a file.

In [4]:
class BCPolicy(nn.Module):
    """The notebook-7 winner: state + k-step history -> action. Small on purpose."""
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, act_dim), nn.Tanh())
    def forward(self, x): return self.net(x)

HISTORY = 1                                 # nb 7: 1 frame captured ~all the gain
OBS_DIM = len(JOINTS) * (HISTORY + 1)
policy = BCPolicy(OBS_DIM, len(JOINTS))

# train briefly on the dataset we just wrote, so the export is of a real policy
s_mean = np.array(STATS["observation.state"]["mean"], np.float32)
s_std  = np.array(STATS["observation.state"]["std"],  np.float32)
a_min  = np.array(STATS["action"]["min"], np.float32)
a_max  = np.array(STATS["action"]["max"], np.float32)
X, Y = [], []
for e in range(N_EPISODES):
    g = dd[dd.episode_index == e]
    st = np.stack(g["observation.state"].values).astype(np.float32)
    ac = np.stack(g["action"].values).astype(np.float32)
    for f in range(len(st)):
        hist = [st[max(0, f - k)] for k in range(HISTORY, -1, -1)]
        X.append(np.concatenate([(h - s_mean) / s_std for h in hist]))
        Y.append(2 * (ac[f] - a_min) / (a_max - a_min) - 1)
X = torch.tensor(np.array(X, np.float32)); Y = torch.tensor(np.array(Y, np.float32))
opt = torch.optim.Adam(policy.parameters(), 1e-3)
for _ in range(1500):
    i = torch.randint(0, len(X), (128,))
    opt.zero_grad(); nn.functional.mse_loss(policy(X[i]), Y[i]).backward(); opt.step()
with torch.no_grad():
    final = nn.functional.mse_loss(policy(X), Y).item()
print(f"trained a {sum(p.numel() for p in policy.parameters()):,}-parameter policy, "
      f"final MSE {final:.5f}")

trained a 70,662-parameter policy, final MSE 0.07455


In [5]:
PKG = EXPORT / "so101_bc_policy"
if PKG.exists(): shutil.rmtree(PKG)
PKG.mkdir(parents=True)

torch.save(policy.state_dict(), PKG / "policy.pt")
sd_hash = hashlib.sha256(open(PKG / "policy.pt", "rb").read()).hexdigest()[:16]

card = {
    "name": "so101_bc_history1",
    "created_with": "physical-ai course, notebook 10",
    "architecture": {"type": "MLP", "hidden": [256, 256], "activation": "ReLU",
                     "output_activation": "tanh",
                     "parameters": sum(p.numel() for p in policy.parameters())},
    "weights": {"file": "policy.pt", "sha256_16": sd_hash},

    # ---- the observation contract: without this the weights are useless ----
    "observation": {
        "description": f"current joint state concatenated with the previous {HISTORY} frame(s)",
        "order": [f"t-{k}" for k in range(HISTORY, -1, -1)],
        "per_frame_features": [f"{j}.pos" for j in JOINTS],
        "dim": OBS_DIM,
        "normalization": "mean/std, per joint, from meta of the training set",
        "state_mean": STATS["observation.state"]["mean"],
        "state_std": STATS["observation.state"]["std"],
        "history_padding": "at episode start, repeat the first frame",
    },
    "action": {
        "semantics": "POSITION TARGETS in normalised servo units, not torques (nb 1)",
        "dim": len(JOINTS),
        "names": [f"{j}.pos" for j in JOINTS],
        "network_output_range": [-1, 1],
        "denormalization": "a = (y+1)/2 * (max-min) + min",
        "action_min": STATS["action"]["min"],
        "action_max": STATS["action"]["max"],
    },
    "control": {
        "rate_hz": FPS,
        "note": "trained at this rate; the MDP changes if you run it at another (nb 1 s5)",
        "expected_actuation_lag_frames": IDENT["validation"]["real_lag_frames"],
    },
    "trained_against_dynamics": IDENT["identified_parameters"],
    "dynamics_provenance": {
        "source_model": IDENT["source_model"],
        "identified_in": "notebook 5",
        "held_out_rmse_vs_real": IDENT["validation"]["identified_rmse_val"],
        "official_model_rmse_vs_real": IDENT["validation"]["official_rmse_val"],
    },
    "limitations": [
        "FREE SPACE ONLY: the dynamics fit used reach-and-carry motion; nothing here "
        "constrains contact, grasping, or pushing (nb 5, nb 9)",
        "no vision: the policy cannot see objects, so it cannot adapt to object placement (nb 7)",
        "trained on a synthetic dataset in this notebook -- NOT a deployable policy, "
        "it exists to demonstrate the export contract",
        "single seed; nb 7 showed offline and closed-loop metrics can rank models in "
        "opposite orders, so offline MSE is weak evidence",
    ],
    "environment": {"python": platform.python_version(), "torch": torch.__version__},
}
json.dump(card, open(PKG / "model_card.json", "w"), indent=2)

# a tiny runnable reference implementation -- the thing that prevents guesswork
REF = "\n".join([
    "# Minimal reference inference for so101_bc_history1.",
    "# Deliberately dependency-light: torch + numpy. Read model_card.json for the contract.",
    "import json, numpy as np, torch, torch.nn as nn",
    "from pathlib import Path",
    "",
    "HERE = Path(__file__).parent",
    "CARD = json.load(open(HERE / 'model_card.json'))",
    "",
    "class BCPolicy(nn.Module):",
    "    def __init__(self, i, o, h=256):",
    "        super().__init__()",
    "        self.net = nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(),",
    "                                 nn.Linear(h, o), nn.Tanh())",
    "    def forward(self, x): return self.net(x)",
    "",
    "_s_mean = np.array(CARD['observation']['state_mean'], np.float32)",
    "_s_std  = np.array(CARD['observation']['state_std'],  np.float32)",
    "_a_min  = np.array(CARD['action']['action_min'], np.float32)",
    "_a_max  = np.array(CARD['action']['action_max'], np.float32)",
    "_n_hist = len(CARD['observation']['order']) - 1",
    "",
    "def load():",
    "    net = BCPolicy(CARD['observation']['dim'], CARD['action']['dim'])",
    "    net.load_state_dict(torch.load(HERE / 'policy.pt', map_location='cpu'))",
    "    net.eval()",
    "    return net",
    "",
    "def act(net, history):",
    "    # history: recent joint-state arrays, OLDEST FIRST. Returns position targets.",
    "    frames = history[-(_n_hist + 1):]",
    "    while len(frames) < _n_hist + 1:          # episode start: repeat the first frame",
    "        frames = [frames[0]] + frames",
    "    x = np.concatenate([(np.asarray(f, np.float32) - _s_mean) / _s_std for f in frames])",
    "    with torch.no_grad():",
    "        y = net(torch.tensor(x)[None]).numpy()[0]",
    "    return (y + 1) / 2 * (_a_max - _a_min) + _a_min",
    "",
    "if __name__ == '__main__':",
    "    net = load()",
    "    s = np.array(CARD['observation']['state_mean'], np.float32)",
    "    print('action for a mean-pose observation:', np.round(act(net, [s, s]), 2))",
    ""])
(PKG / "run_policy.py").write_text(REF)
print(f"exported to {PKG.relative_to(EXPORT.parent)}:")
for p in sorted(PKG.iterdir()):
    print(f"  {p.name:18} {p.stat().st_size/1024:7.1f} KB")

exported to export/so101_bc_policy:
  model_card.json        2.9 KB
  policy.pt            278.7 KB
  run_policy.py          1.7 KB


In [6]:
# Verify the export is self-contained by running the reference script in a FRESH
# process that imports nothing from this notebook.
#
# Note: pass an ABSOLUTE path. My first version passed a relative path together with
# cwd=PKG, so the path resolved against the new cwd and the file "did not exist" --
# the harness failed, not the export. Worth the comment, because a self-containment
# check that fails for its own bug is worse than no check at all.
import subprocess, sys
script = (PKG / "run_policy.py").resolve()
r = subprocess.run([sys.executable, str(script)], capture_output=True, text=True,
                   cwd=str(PKG.resolve()))
print("stdout:", r.stdout.strip())
if r.returncode != 0:
    print("stderr:", r.stderr.strip()[-700:])
print(f"\nreference implementation runs standalone: {r.returncode == 0}")
print("That is the bar for an export: a fresh process, the files, and nothing else.")

stdout: action for a mean-pose observation: [  0.29 -47.53  68.56  74.55 -54.73   5.08]

reference implementation runs standalone: True
That is the bar for an export: a fresh process, the files, and nothing else.


## 4. The deployment checklist

Everything in this course that can be verified **without** a robot, in the order you
should do it. Each line traces to a notebook where skipping it caused a real failure.

| # | Check | From |
|---|---|---|
| 1 | Is the task physically possible? Compute the workspace and confirm the target is inside it | nb 4, and `scene_box.xml`'s box at x=0.5 vs 0.405 reach |
| 2 | Are your reward terms on comparable scales? Print each term's per-step magnitude | **nb 8 bug #1** |
| 3 | Is `terminated` distinguished from `truncated` in your Bellman target? | **nb 8 bug #2** |
| 4 | Does the actuator saturate at your operating error? Log actuator force, not just response | nb 4 (saturates at 0.17°) |
| 5 | Does your sim reproduce a *measured* property of the real robot? | nb 5 (lag 4 frames) |
| 6 | Have you evaluated **closed-loop**, not just teacher-forced? | nb 6 (1.65 vs 35.4) |
| 7 | Does your offline metric rank candidates the same way closed-loop does? | **nb 7 (corr = −0.25)** |
| 8 | Do your control and observation rates match the real robot? | nb 1, nb 3 |
| 9 | Is your policy's export self-contained? Run it in a fresh process | this notebook |
| 10 | Multiple seeds before you believe a ranking | nb 7, nb 8 |

Item 7 is the one I would put first if forced to choose. Notebook 7 measured offline MSE
to be *negatively* correlated with closed-loop performance across twelve variants — the
best policy had 5.6× worse validation loss than the model validation loss preferred. An
offline number is not weak evidence of deployment performance; on that evidence it is
mildly misleading evidence.

In [7]:
# Run the parts of the checklist that are mechanically checkable right now.
import mujoco
results = []

m = mujoco.MjModel.from_xml_path("../models/so101/so101_identified.xml")
TIP = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, "moving_jaw_so101_v1")
d = mujoco.MjData(m); rng = np.random.default_rng(0); pts = []
for _ in range(2000):
    d.qpos[:] = rng.uniform(m.jnt_range[:, 0], m.jnt_range[:, 1])
    mujoco.mj_forward(m, d); pts.append(d.xpos[TIP].copy())
P = np.array(pts); reach = float(np.linalg.norm(P[:, :2], axis=1).max())
results.append(("1. workspace computed", True, f"max planar reach {reach*100:.1f} cm"))

# the shipped box scene: is its target inside that workspace?
box_x = 0.5
results.append(("1b. scene_box.xml target reachable", box_x <= P[:, 0].max(),
                f"box x={box_x} m vs max tip x={P[:,0].max():.3f} m"))

# Saturation threshold = forcerange / kp. Identification MOVED this: the official
# kp=998 saturated at 0.17 deg (nb 4), so the actuator was effectively bang-bang.
# The identified kp=18 saturates at ~9 deg, i.e. it operates in its LINEAR region for
# realistic tracking errors -- which is why the fitted model reproduces the real arm's
# speed-dependent lag instead of a constant one.
sat_ident = np.rad2deg(m.actuator_forcerange[0, 1] / m.actuator_gainprm[0, 0])
m_off = mujoco.MjModel.from_xml_path("../models/so101/so101.xml")
sat_off = np.rad2deg(m_off.actuator_forcerange[0, 1] / m_off.actuator_gainprm[0, 0])
results.append(("4. saturation threshold known", True,
                f"identified {sat_ident:.2f} deg vs official {sat_off:.2f} deg"))
results.append(("5. sim validated against a measurement", True,
                f"lag {IDENT['validation']['identified_lag_frames']:.0f} == real "
                f"{IDENT['validation']['real_lag_frames']}"))
results.append(("8. control rate recorded in the export", card["control"]["rate_hz"] == FPS,
                f"{card['control']['rate_hz']} Hz"))
results.append(("9. export self-contained", r.returncode == 0, "fresh-process run"))
results.append(("Export declares its limitations", len(card["limitations"]) >= 3,
                f"{len(card['limitations'])} listed"))

print(f"{'check':42} {'status':>8}  detail")
print("=" * 86)
for name, ok, detail in results:
    print(f"{name:42} {'PASS' if ok else 'FAIL':>8}  {detail}")
print("=" * 86)
fails = [n for n, ok, _ in results if not ok]
print(f"\n{len(fails)} failing: {fails}" if fails else "\nall mechanical checks pass")

check                                        status  detail
1. workspace computed                          PASS  max planar reach 40.5 cm
1b. scene_box.xml target reachable             FAIL  box x=0.5 m vs max tip x=0.405 m
4. saturation threshold known                  PASS  identified 9.36 deg vs official 0.17 deg
5. sim validated against a measurement         PASS  lag 4 == real 4
8. control rate recorded in the export         PASS  30 Hz
9. export self-contained                       PASS  fresh-process run
Export declares its limitations                PASS  4 listed

1 failing: ['1b. scene_box.xml target reachable']


### The failing check is the interesting one

One check fails, for a real reason.

`scene_box.xml` — shipped with the official Menagerie model — places its box at
**x = 0.5 m**, and this arm's maximum tip reach is **0.405 m**. The box is outside the
workspace.

That matters beyond this one file. A policy trained to reach it would produce a reward
curve that never improves, and **that looks exactly like a broken algorithm**. Notebook 8
burned real time on a reward-scale bug and a Bellman-mask bug that produced the same flat
curve. Confirming the task is possible costs thirty seconds and eliminates a whole class
of misdiagnosis — which is why it is item 1 on the checklist rather than buried under the
modelling items.

### A number worth reading twice

The saturation row reports **9.36° for the identified model against 0.17° for the
official one**. Notebook 4 found that with `kp` = 998 the actuator saturates at a 0.17°
error, making it a bang-bang maximum-torque source for any realistic motion — which is
why its step response measured the force limit rather than the gain.

Identification moved that threshold by a factor of ~55. At `kp` = 18 the actuator spends
realistic motion **inside its linear region**, which is precisely why the fitted model
reproduces the real arm's *speed-dependent* tracking error (notebook 5's independent
check: correlation 0.977 against the real 0.885) rather than a constant lag. The two
findings are the same fact seen from two directions.

## 5. The LeRobot environment, and the hardware path

Now the part that needs a separate environment and, eventually, a robot.

### A separate environment, on purpose

```bash
# LeRobot pins torch 2.10; our course needs 2.14. Separate environments, shared files.
uv venv --python 3.11 .venv-lerobot
uv pip install --python .venv-lerobot/bin/python "lerobot[feetech]"
```

The `feetech` extra is the one that matters for an SO-101 — those are the STS3215 servo
bus drivers. The two environments exchange **LeRobot-format datasets and policy
directories**, which is precisely what the format exists for.

### From the dataset we wrote to a real robot

```bash
L=.venv-lerobot/bin/python

# 1. find the arm's serial port
$L -m lerobot.find_port

# 2. calibrate both arms -- do NOT skip this. Every normalisation
#    statistic in our export assumes the calibration it was recorded under.
$L -m lerobot.calibrate --robot.type=so101_follower --robot.port=/dev/tty.XXXX
$L -m lerobot.calibrate --teleop.type=so101_leader  --teleop.port=/dev/tty.YYYY

# 3. teleoperate, and record real demonstrations in the format from section 1
$L -m lerobot.record --robot.type=so101_follower --robot.port=/dev/tty.XXXX \
      --teleop.type=so101_leader --teleop.port=/dev/tty.YYYY \
      --dataset.repo_id=$USER/so101_pickplace --dataset.num_episodes=50

# 4. train a real policy (ACT or diffusion -- notebook 7's chunking methods)
$L -m lerobot.scripts.train --dataset.repo_id=$USER/so101_pickplace \
      --policy.type=act --output_dir=outputs/act_so101

# 5. evaluate on the robot
$L -m lerobot.record --robot.type=so101_follower --robot.port=/dev/tty.XXXX \
      --policy.path=outputs/act_so101/checkpoints/last/pretrained_model \
      --dataset.repo_id=$USER/eval_act --dataset.num_episodes=10
```

Treat these as the current shape of the CLI rather than a guarantee — LeRobot moves
quickly, and the flags have changed more than once. Check `--help` before relying on any
of them.

### Things this course taught that the CLI will not tell you

- **Calibrate before recording, and never re-calibrate mid-dataset.** Notebook 2's
  normalization statistics, and the whole unit mapping in notebook 5, assume one fixed
  calibration. Changing it silently invalidates every number downstream.
- **Record a second past the release.** Notebook 2 found only **26 of 50** episodes in
  the official dataset contained a completed release — the operator stopped the take too
  early. A policy trained on that is systematically worse at finishing than starting.
- **Order spare STS3215 servos with the kit.** They fail, and a result you cannot
  reproduce next week may be mechanical rather than software.
- **Clamp the arm.** An unclamped SO-101 walks itself off the bench.
- **Video every run from the first day.** It is your only real log; you cannot
  print-statement a grasp that slips.

## 6. Course summary — what Part IV was for

**What this notebook produced:**

1. A **schema-valid LeRobot v3 dataset**, validated by a checker *and* by notebook 1's
   lag analysis running on it unmodified and recovering the lag we injected.
2. A **self-contained policy export** — weights, observation contract, both sets of
   normalization statistics, action semantics, control rate, the identified dynamics it
   assumes, and an explicit limitations list — verified by running its reference
   implementation in a fresh subprocess.
3. A **deployment checklist** where every item traces to a specific failure earlier in
   this course, with one item currently failing (the unreachable box) for a genuine reason.

**The through-line of the whole course:** every notebook's useful result came from a
measurement that contradicted something plausible.

| notebook | the plausible thing | what measurement said |
|---|---|---|
| 1 | an action is a torque | it is a position target, 4 frames of lag, 50/50 episodes |
| 2 | robot data is multimodal | **not this dataset** — 0.0% of gaps exceed the baseline |
| 4 | a 30° step measures stiffness | it saturates at 0.17°, measuring the force limit |
| 5 | the official model describes the robot | 0-frame lag vs the real 4; RMSE 6.19 → 1.44 |
| 6 | BC averaging makes the policy freeze | it moves at 97% speed and goes the wrong way |
| 7 | chunking helps because data is multimodal | it helps via feedback rate, and offline metrics rank it **negatively** |
| 8 | SAC needed more samples | two reward/bootstrap bugs; 29.6 → 2.0 cm |

Seven notebooks, seven corrections. The reason to keep them visible rather than tidy them
away is that the corrections are the transferable part — the specific numbers are not.

### Final exercises

1. **Write your own dataset from notebook 8's SAC rollouts.** Replace
   `make_rollout` with real policy rollouts from the identified simulator and export those
   as a LeRobot dataset. You then have simulated demonstrations in the same format as real
   ones, which is how sim-to-real data pipelines actually work.
2. **Add a camera.** Extend the writer to emit MP4 alongside the Parquet using
   MuJoCo's offscreen renderer. Then notebook 7's vision experiment can be run
   closed-loop, which our object-free simulator could not support.
3. **Make the export loadable by LeRobot.** Convert the policy directory to LeRobot's
   `pretrained_model` layout in the separate environment, and confirm
   `lerobot.record --policy.path=...` accepts it. That closes the last gap between this
   course and the production stack.
4. **Re-run the checklist on someone else's project.** Pick any robot-learning repo and
   work through the ten items. The ones it fails will tell you more about it in an hour
   than reading the code for a day.